In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


df = pd.read_csv('rahul_transactions.csv')
print("Original shape:", df.shape)


def clean_date(x):
    x = str(x).strip()
    for fmt in ['%Y-%m-%d', '%d/%m/%y', '%d-%b-%y', '%d %b %Y', '%d-%m-%y']:
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            pass
    return pd.to_datetime(x, dayfirst=True, errors='coerce')

df['date'] = df['Date'].apply(clean_date)

temp = df['Amount'].astype(str)
temp = temp.str.replace('₹', '')
temp = temp.str.replace('Rs.', '')
temp = temp.str.replace(',', '')
temp = temp.str.strip()
df['amount'] = pd.to_numeric(temp, errors='coerce')

df['txn_type'] = df['Type'].str.lower()
df['txn_type'] = df['txn_type'].replace({'dr': 'debit', 'cr': 'credit'})

print("Duplicates found:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print("After cleaning:", df.shape)
print("Date range:", df['date'].min().date(), "to", df['date'].max().date())
print("Null dates:", df['date'].isna().sum())
print("Null amounts:", df['amount'].isna().sum())

#vendor extractor

vendor_dict = {
    'Swiggy': ['SWIGGY', 'BUNDL'],
    'Zomato': ['ZOMATO'],
    'Zepto': ['ZEPTO'],
    'Blinkit': ['BLINKIT', 'GROFERS'],
    'Instamart': ['INSTAMART'],
    'Amazon': ['AMAZON', 'AMZN'],
    'Flipkart': ['FLIPKART', 'FKART', 'FSN'],
    'Myntra': ['MYNTRA'],
    'Nykaa': ['NYKAA'],
    'Uber': ['UBER', 'ANI TECHNOLOGIES'],
    'Ola': ['OLA'],
    'Rapido': ['RAPIDO', 'ROPPEN'],
    'BMTC': ['BMTC', 'TUMMOC'],
    'Starbucks': ['STARBUCKS'],
    'Cafe Coffee Day': ['COFFEE DAY', 'CCD'],
    'Third Wave': ['THIRD WAVE', 'THIRDWAVE', 'TWC'],
    'Truffles': ['TRUFFLES'],
    'Meghana': ['MEGHANA'],
    'Empire': ['EMPIRE'],
    'Netflix': ['NETFLIX'],
    'Spotify': ['SPOTIFY'],
    'Hotstar': ['HOTSTAR', 'DISNEY'],
    'BookMyShow': ['BOOKMYSHOW', 'BMS', 'BIGTREE'],
    'BESCOM': ['BESCOM', 'ELEC SUPPLY', 'BANGALORE ELEC'],
    'BWSSB': ['BWSSB', 'WATER'],
    'Airtel': ['AIRTEL', 'BHARTI'],
    'Jio': ['JIO', 'RELIANCE JIO'],
    'VI': ['VODAFONE', 'VI POSTPAID', 'VI-RECHARGE'],
    'BigBasket': ['BIGBASKET'],
    'DMart': ['DMART', 'AVENUE SUPERMARTS'],
    'Zerodha': ['ZERODHA'],
    'Groww': ['GROWW'],
    'Petrol': ['PETROL', 'BPCL', 'HP PETROL', 'INDIAN OIL', 'IOC'],
    'Rent': ['RENT-LANDLORD', 'LANDLORD'],
    'P2P': ['UPI-AMAN', 'UPI-ANKIT', 'UPI-VIKAS', 'UPI-PRIYA', 'UPI-KARAN', 'UPI-NEHA', 'UPI-SNEHA'],
    'ATM': ['ATM-WDL'],
    'Salary': ['TECHCRUSH', 'SALARY']
}

def get_vendor(desc):
    desc = str(desc).upper()
    for vendor, keys in vendor_dict.items():
        for k in keys:
            if k in desc:
                return vendor
    return 'Other'

df['vendor'] = df['Description'].apply(get_vendor)
print("\nTop vendors:")
print(df['vendor'].value_counts().head(12))

#category tagger

cat_map = {
    'Swiggy': 'Food Delivery',
    'Zomato': 'Food Delivery',
    'Zepto': 'Quick Commerce',
    'Blinkit': 'Quick Commerce',
    'Instamart': 'Quick Commerce',
    'Amazon': 'E-commerce',
    'Flipkart': 'E-commerce',
    'Myntra': 'E-commerce',
    'Nykaa': 'E-commerce',
    'Uber': 'Transport',
    'Ola': 'Transport',
    'Rapido': 'Transport',
    'BMTC': 'Transport',
    'Starbucks': 'Cafe',
    'Cafe Coffee Day': 'Cafe',
    'Third Wave': 'Cafe',
    'Truffles': 'Restaurants',
    'Meghana': 'Restaurants',
    'Empire': 'Restaurants',
    'Netflix': 'Subscriptions',
    'Spotify': 'Subscriptions',
    'Hotstar': 'Subscriptions',
    'BookMyShow': 'Entertainment',
    'BESCOM': 'Utilities',
    'BWSSB': 'Utilities',
    'Airtel': 'Utilities',
    'Jio': 'Utilities',
    'VI': 'Utilities',
    'BigBasket': 'Groceries',
    'DMart': 'Groceries',
    'Zerodha': 'Investments',
    'Groww': 'Investments',
    'Petrol': 'Fuel',
    'Rent': 'Personal Transfer',
    'P2P': 'Personal Transfer',
    'ATM': 'Cash Withdrawal',
    'Salary': 'Income',
    'Other': 'Other'
}

df['category'] = df['vendor'].map(cat_map)
print("\nCategory counts:")
print(df['category'].value_counts())

#spending overview

debit_df = df[df['txn_type'] == 'debit'].copy()
credit_df = df[df['txn_type'] == 'credit'].copy()

total_credit = credit_df['amount'].sum()
total_debit = debit_df['amount'].sum()
net = total_credit - total_debit
savings = (net / total_credit) * 100

print("\n" + "="*60)
print("EXECUTIVE SUMMARY")
print("="*60)
print(f"Total Credits    : Rs {total_credit:,.0f}")
print(f"Total Debits     : Rs {total_debit:,.0f}")
print(f"Net Change       : Rs {net:,.0f}")
print(f"Savings Rate     : {savings:.1f}%")
print(f"Total txns       : {len(df)}")
print(f"Unique vendors   : {df['vendor'].nunique()}")

cat_totals = debit_df.groupby('category')['amount'].sum().sort_values(ascending=False)
print("\nTOP CATEGORIES")
print("-"*40)
for c, a in cat_totals.head(6).items():
    pct = (a / total_debit) * 100
    print(f"{c:20} {pct:5.1f}%   Rs {a:,.0f}")

vend_totals = debit_df.groupby('vendor').agg(
    spent=('amount', 'sum'),
    count=('amount', 'count')
).sort_values('spent', ascending=False)

print("\nTOP VENDORS")
print("-"*40)
for v, row in vend_totals.head(6).iterrows():
    print(f"{v:15} Rs {row['spent']:>9,.0f}  ({int(row['count'])} orders)")

#monthly trend

debit_df['month'] = debit_df['date'].dt.month_name().str[:3]
monthly = debit_df.pivot_table(index='category', columns='month', values='amount', aggfunc='sum', fill_value=0)
months_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
monthly = monthly.reindex(columns=[m for m in months_order if m in monthly.columns])

print("\nFood Delivery month-wise:")
if 'Food Delivery' in monthly.index:
    print(monthly.loc['Food Delivery'])

#time of day

debit_df['hour'] = debit_df['Time'].astype(str).str[:2].astype(int)
food = debit_df[debit_df['category'] == 'Food Delivery']
late = food[(food['hour'] >= 21) | (food['hour'] <= 2)]

print(f"\nFood Delivery total orders : {len(food)}")
print(f"Late night (9pm-2am)       : {len(late)}")
print(f"Late night percentage      : {round(len(late)/len(food)*100, 1)}%")

#anomaly detection

debit_df['mean_amt'] = debit_df.groupby('category')['amount'].transform('mean')
debit_df['std_amt'] = debit_df.groupby('category')['amount'].transform('std')
debit_df['z'] = (debit_df['amount'] - debit_df['mean_amt']) / debit_df['std_amt']

anoms = debit_df[debit_df['z'] > 2].sort_values('z', ascending=False)

print(f"\nAnomalies found (z > 2): {len(anoms)}")
print(anoms[['date', 'vendor', 'category', 'amount', 'z']].head(6))

# ====================== ARCHETYPES ======================
def check_archetypes():
    result = []
    food_pct = debit_df[debit_df['category'].isin(['Food Delivery', 'Restaurants', 'Cafe'])]['amount'].sum() / total_debit * 100
    if food_pct > 25:
        result.append(f"THE FOODIE ({food_pct:.1f}% on food)")

    qc_pct = debit_df[debit_df['category'] == 'Quick Commerce']['amount'].sum() / total_debit * 100
    if qc_pct > 15:
        result.append(f"THE QUICK COMMERCE JUNKIE ({qc_pct:.1f}%)")

    ecom_pct = debit_df[debit_df['category'] == 'E-commerce']['amount'].sum() / total_debit * 100
    if ecom_pct > 15:
        result.append(f"THE SHOPAHOLIC ({ecom_pct:.1f}% on shopping)")

    inv_pct = debit_df[debit_df['category'] == 'Investments']['amount'].sum() / total_debit * 100
    if inv_pct > 15:
        result.append(f"THE INVESTOR ({inv_pct:.1f}% on investments)")

    if len(food) > 0:
        late_pct = len(late) / len(food) * 100
        if late_pct > 50:
            result.append(f"THE LATE-NIGHT SNACKER ({late_pct:.0f}% food after 9pm)")

    if savings < 10:
        result.append(f"THE YOLO SPENDER (savings rate {savings:.1f}%)")

    return result

arch = check_archetypes()

print("\nDetected Archetypes:")
for a in arch:
    print("→", a)

#final rpeort
print("\n" + "="*60)
print("SpendDNA REPORT - RAHUL SHARMA")
print(f"Period: Jan 2024 to Jun 2024 | {len(df)} transactions")
print("="*60)

print("\nEXECUTIVE SUMMARY")
print("-"*40)
print(f"Total Credits    : Rs {total_credit:,.0f}")
print(f"Total Debits     : Rs {total_debit:,.0f}")
print(f"Net Change       : Rs {net:,.0f}")
print(f"Savings Rate     : {savings:.1f}%")
print(f"Unique Vendors   : {df['vendor'].nunique()}")

print("\nTOP CATEGORIES")
print("-"*40)
for c, a in cat_totals.head(5).items():
    pct = a / total_debit * 100
    print(f"{c:18} {pct:5.1f}%   Rs {a:,.0f}")

print("\nTOP VENDORS")
print("-"*40)
for v, row in vend_totals.head(5).iterrows():
    print(f"{v:15} Rs {row['spent']:>9,.0f}  ({int(row['count'])} orders)")

print("\nTIME PATTERNS")
print("-"*40)
print(f"Food Delivery late night share: {len(late)/len(food)*100:.0f}%")

print("\nTOP ANOMALIES")
print("-"*40)
for i, row in anoms.head(5).iterrows():
    print(f"{row['date'].strftime('%d %b')}  {row['vendor']:12}  Rs {row['amount']:>8,.0f}  (z={row['z']:.1f})")

print("\nSPENDING ARCHETYPES")
print("-"*40)
for a in arch:
    print("→", a)



Original shape: (1328, 8)
Duplicates found: 18
After cleaning: (1310, 11)
Date range: 2024-01-01 to 2024-06-30
Null dates: 0
Null amounts: 0

Top vendors:
vendor
Swiggy       223
Zomato       121
Uber          89
Amazon        86
Ola           69
Other         63
Zepto         58
Rapido        55
Blinkit       55
Flipkart      50
Starbucks     42
BMTC          37
Name: count, dtype: int64

Category counts:
category
Food Delivery        344
Transport            250
E-commerce           172
Quick Commerce       133
Cafe                  99
Other                 63
Utilities             43
Restaurants           34
Groceries             33
Fuel                  28
Subscriptions         28
Personal Transfer     24
Investments           23
Cash Withdrawal       17
Entertainment         13
Income                 6
Name: count, dtype: int64

EXECUTIVE SUMMARY
Total Credits    : Rs 509,774
Total Debits     : Rs 1,678,901
Net Change       : Rs -1,169,127
Savings Rate     : -229.3%
Total txns    